In [1]:
import os

# do not do:
# !cd
# !cd react
# !cd
# !git fetch --all --tags

# beacause:
# The !cd command changes the directory only for that specific cell's execution context. However, the next ! command runs in a new process, so the directory change does not persist between commands.
# 
# To ensure the directory change persists across commands, it is needed to use Python's built-in os module 
print("Current Working Directory:", os.getcwd())
# Change to the react directory
os.chdir("./react")

print("Current Working Directory:", os.getcwd())
# Verify the current working directory


Current Working Directory: C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution
Current Working Directory: C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react


In [2]:
import subprocess

# Get the list of commits between v17.0.1 and v17.0.2
commit_range = 'v17.0.1..v17.0.2'

# Run the git command
result = subprocess.run(['git', 'log', commit_range, '--pretty=format:%H %s'], capture_output=True, text=True)

# Check for errors
if result.returncode != 0:
    print("Error running git command:", result.stderr)
else:
    commits_output = result.stdout.strip()
    print("Commits between v17.0.1 and v17.0.2:")
    print(commits_output)
    
# ------------------------ ALTERNATIVE ------------------------
# ------------------------ YOU CAN RUN THIS CODE ------------------------
# List all commits between the two versions
# Run git command
# !git fetch --all --tags
# !git log v17.0.1..v17.0.2 --oneline

Commits between v17.0.1 and v17.0.2:
12adaffef7105e2714f82651ea51936c563fe15c Remove scheduler sampling profiler shared array buffer (#20840)
b2bbee7ba31bb7d212a9ff2e682a695a32f8a87f Disable (unstable) scheduler sampling profiler for OSS builds (#20832)
8cc6ff24880ac00fdb9d11bce480a0433456e82d fix: use SharedArrayBuffer only when cross-origin isolation is enabled (#20831)


In [3]:
# Parse the commits_output to get a list of commit hashes
commits = []
for line in commits_output.split('\n'):
    commit_hash = line.split()[0]
    commits.append(commit_hash)

# Print the list of commit hashes
print("Commit Hashes:")
for commit in commits:
    print(commit)


Commit Hashes:
12adaffef7105e2714f82651ea51936c563fe15c
b2bbee7ba31bb7d212a9ff2e682a695a32f8a87f
8cc6ff24880ac00fdb9d11bce480a0433456e82d


In [4]:
# Initialize variables to track the commit with the most substantial change
max_files_changed = 0
max_commit_hash = ''
max_insertions = 0
max_deletions = 0

# Analyze each commit
for commit in commits:
    # Run git show to get the stats for the commit
    result = subprocess.run(['git', 'show', '--numstat', '--format=', commit], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running git show for commit {commit}:", result.stderr)
        continue

    numstat_output = result.stdout.strip()
    files_changed = 0
    insertions = 0
    deletions = 0

    # Parse the numstat output
    for line in numstat_output.split('\n'):
        if line:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                insertions_line, deletions_line, _ = parts
                try:
                    insertions += int(insertions_line) if insertions_line != '-' else 0
                    deletions += int(deletions_line) if deletions_line != '-' else 0
                    files_changed += 1
                except ValueError:
                    # Handle the case where insertions or deletions are '-'
                    pass

    # Check if this commit has the most files changed
    if files_changed > max_files_changed:
        max_files_changed = files_changed
        max_commit_hash = commit
        max_insertions = insertions
        max_deletions = deletions

# Print the commit with the most substantial change
print("\nCommit with the most substantial change:")
print(f"Commit Hash: {max_commit_hash}")
print(f"Files Changed: {max_files_changed}")
print(f"Insertions: {max_insertions}")
print(f"Deletions: {max_deletions}")




Commit with the most substantial change:
Commit Hash: 12adaffef7105e2714f82651ea51936c563fe15c
Files Changed: 4
Insertions: 15
Deletions: 123


Or define single functions:

In [5]:
def get_commit_list(commit_range):
    result = subprocess.run(['git', 'log', commit_range, '--pretty=format:%H %s'], capture_output=True, text=True)
    if result.returncode != 0:
        print("Error running git command:", result.stderr)
        return []
    commits_output = result.stdout.strip()
    commits = []
    for line in commits_output.split('\n'):
        commit_hash = line.split()[0]
        commits.append(commit_hash)
    return commits

def analyze_commits(commits):
    max_files_changed = 0
    max_commit_hash = ''
    max_insertions = 0
    max_deletions = 0

    for commit in commits:
        result = subprocess.run(['git', 'show', '--numstat', '--format=', commit], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Error running git show for commit {commit}:", result.stderr)
            continue

        numstat_output = result.stdout.strip()
        files_changed = 0
        insertions = 0
        deletions = 0

        for line in numstat_output.split('\n'):
            if line:
                parts = line.strip().split('\t')
                if len(parts) == 3:
                    insertions_line, deletions_line, _ = parts
                    try:
                        insertions += int(insertions_line) if insertions_line != '-' else 0
                        deletions += int(deletions_line) if deletions_line != '-' else 0
                        files_changed += 1
                    except ValueError:
                        pass

        if files_changed > max_files_changed:
            max_files_changed = files_changed
            max_commit_hash = commit
            max_insertions = insertions
            max_deletions = deletions

    return max_commit_hash, max_files_changed, max_insertions, max_deletions

def get_commit_details(commit_hash):
    result = subprocess.run(['git', 'show', '--stat', commit_hash], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running git show for commit {commit_hash}:", result.stderr)
        return ''
    return result.stdout.strip()


In [6]:
# Define the commit range
commit_range = 'v17.0.1..v17.0.2'

# Get the list of commits
commits = get_commit_list(commit_range)
print(f"Found {len(commits)} commits in the range {commit_range}")

# Analyze the commits
max_commit_hash, max_files_changed, max_insertions, max_deletions = analyze_commits(commits)

# Print the result
print("\nCommit with the most substantial change:")
print(f"Commit Hash: {max_commit_hash}")
print(f"Files Changed: {max_files_changed}")
print(f"Insertions: {max_insertions}")
print(f"Deletions: {max_deletions}")

# Get and print the commit details
commit_details = get_commit_details(max_commit_hash)
print("\nCommit Details:")
print(commit_details)


Found 3 commits in the range v17.0.1..v17.0.2

Commit with the most substantial change:
Commit Hash: 12adaffef7105e2714f82651ea51936c563fe15c
Files Changed: 4
Insertions: 15
Deletions: 123

Commit Details:
commit 12adaffef7105e2714f82651ea51936c563fe15c
Author: Brian Vaughn <bvaughn@fb.com>
Date:   Thu Feb 18 11:21:52 2021 -0500

    Remove scheduler sampling profiler shared array buffer (#20840)
    
    No one has been using this data so there's no reason to collect it. Event log has been maintained and tests have been updated.

 packages/scheduler/src/Scheduler.js                |  2 -
 packages/scheduler/src/SchedulerFeatureFlags.js    |  2 +-
 packages/scheduler/src/SchedulerProfiling.js       | 56 ----------------
 .../src/__tests__/SchedulerProfiling-test.js       | 78 ++++------------------
 4 files changed, 15 insertions(+), 123 deletions(-)
